In [ ]:
import pandas as pd
import datetime as dt
from datetime import timezone, timedelta

from rockyclickup.wrapper import Session as rcu_session
from rockyclickup.models import (
    FSA,
    DCA,
    HSA,
    HRA,
    PKG,
    TRN,
    LSA,
    ADO,
    EDU,
)
from rockyclickup.utils import response_to_dataframe as rcu_res_2_df
from rockyclickup.database_interface import get_field_by_name

from rockyelevate.wrapper import Session as elv_session
from rockyelevate.utils import response_to_dataframe as elv_res_2_df


In [ ]:
clickup = rcu_session()
elv = elv_session("PROD", multithread=True, max_threads=40)

In [ ]:
plan_models = [
    FSA,
    DCA,
    HSA,
    HRA,
    PKG,
    TRN,
    LSA,
    ADO,
    EDU,
]

plan_responses = []
for model in plan_models:
    full_list = clickup.get_full_list(model=model)
    plan_responses.extend(full_list)


In [ ]:
plan_df = rcu_res_2_df(plan_responses)

In [ ]:
date_cols = [c for c in plan_df.columns if "date" in c]

wrong_ids = []

for col in date_cols:
    filtered = plan_df[
        plan_df[col].apply(lambda x: isinstance(x, dt.datetime) and x < dt.datetime(2000, 1, 1))
    ]

    if filtered.empty:
        continue

    print(f"{len(filtered):>3} filtered rows for column '{col}'")

    wrong_ids.extend(filtered['id'].to_list())
    


In [ ]:
plans_to_correct = plan_df[plan_df['id'].isin(wrong_ids)]

plans_to_correct[['id', 'elv_id', 'elv_plan_code', 'date_plan_start', 'date_plan_end', 'date_admin_start']]

elv_ids = plans_to_correct['elv_id'].to_list()

elv_response = elv.get_plans_by_id(pids=elv_ids)


In [ ]:
elv_plan_df = elv_res_2_df(elv_response)
for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    elv_plan_df[col] = pd.to_datetime(elv_plan_df[col])

valid_from_map = {
    r.get("id"): r.get("plan_year.valid_from")
    for _,r in elv_plan_df.iterrows()
}

valid_to_map = {
    r.get("id"): r.get("plan_year.valid_to")
    for _,r in elv_plan_df.iterrows()
}


In [ ]:
plans_to_correct['elv_id']

In [ ]:
plans_to_correct['elv_id'] = plans_to_correct['elv_id'].astype(int)
plans_to_correct['true_start_date'] = plans_to_correct['elv_id'].map(valid_from_map)
plans_to_correct['true_end_date'] = plans_to_correct['elv_id'].map(valid_to_map)

In [ ]:
date_plan_start_field_id = get_field_by_name("date_plan_start").field_id
date_plan_end_field_id = get_field_by_name("date_plan_end").field_id

In [ ]:
true_start_date

In [ ]:
my_time = dt.datetime.fromtimestamp(timestamp=1764547200.0)

In [ ]:
my_time

In [ ]:
print(plans_to_correct.iloc[0].get("true_start_date"))
print(plans_to_correct.iloc[0].get("true_start_date").timestamp())
print(dt.datetime.fromtimestamp(
    1761955200, 
    tz=timezone.utc
))


In [ ]:
print(plans_to_correct.iloc[0].get("true_start_date"))
print(plans_to_correct.iloc[0].get("true_start_date").timestamp())
print(dt.datetime.fromtimestamp(
    plans_to_correct.iloc[0]["true_start_date"].timestamp(), 
    tz=timezone.utc
))


In [ ]:


local_offset = timedelta(hours=7)

for index, row in plans_to_correct.iterrows():
    task_id = row.get("id")
    print(task_id)

    true_start_date = int((row["true_start_date"].timestamp() + local_offset.total_seconds()) * 1000)
    true_end_date = int((row["true_end_date"].timestamp() + local_offset.total_seconds()) * 1000)

    print(true_start_date)
    print(true_end_date)

    update_start_res = clickup.patch(
        task_id=task_id,
        field_id=date_plan_start_field_id,
        value=true_start_date
    )

    update_end_res = clickup.patch(
        task_id=task_id,
        field_id=date_plan_end_field_id,
        value=true_end_date
    )

